# Linear Regression

---

## Introduction

In the Perceptron notebook, we built a model that answered a **yes or no question** — is this dog breed long-lived or short-lived? That type of problem is called **classification**.

But what if the answer isn't yes or no? What if we want to predict an actual **number**?

That's a **regression** problem — and **Linear Regression** is the simplest and most fundamental algorithm for solving it. The goal is to find the best straight line through your data:

$$\hat{y} = w \cdot x + b$$

Where $w$ is the slope, $b$ is the intercept, $x$ is the input, and $\hat{y}$ is the predicted output. The model learns $w$ and $b$ by minimizing its mistakes over the training data.

In this notebook we will use Linear Regression to answer one question:

> *Can we predict a state's winter bee colony loss percentage based on how many colonies it has?*

**Dataset:** Bee Colony Loss — compiled by the **Bee Informed Partnership (BIP)** and the **USDA Agricultural Statistics Service**  
**Source:** BIP annual colony loss reports (2010–2021)

**Feature used:** number of colonies per state  
**Target:** winter colony loss percentage

In [ ]:
# ── Standard libraries ───────────────────────────────────────────────────────
import numpy as np        # numerical computing — arrays, math operations
import pandas as pd       # data manipulation — loading and cleaning our dataset
import matplotlib.pyplot as plt  # plotting — visualizing our data and results
import seaborn as sns     # prettier plots built on top of matplotlib

# ── Our custom LinearRegression from the package ─────────────────────────────
import sys
sys.path.insert(0, '../Python Package')
from linear_regression import LinearRegression

# ── Plot styling ──────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('All libraries loaded successfully!')

In [ ]:
# Load the dataset
df = pd.read_csv('data/Bee_Colony_Loss.csv')

# First look at the data
print('Dataset shape:', df.shape)
print()
df.head()

## Step 1 — Understanding the Data

We have **568 rows** of state-level bee colony data across multiple years.

Before we do any modeling we need to understand our data:
- What does each column mean?
- Are there any missing values or formatting issues?
- What does the distribution of our target variable (winter loss %) look like?

In [ ]:
# Check column names and data types
print('Columns and data types:')
print(df.dtypes)

In [ ]:
# Check for missing values in each column
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
# The numeric columns are stored as strings because they contain
# symbols like '%' and ','. Let's clean them and take a first look.

# Remove '%' and convert winter loss to float
df['Loss'] = df['Total Winter All Loss'].str.replace('%', '').str.strip()
df['Loss'] = pd.to_numeric(df['Loss'], errors='coerce')

# Remove commas and convert colony count to float
df['Colonies_Clean'] = df['Colonies'].str.replace(',', '').str.strip()
df['Colonies_Clean'] = pd.to_numeric(df['Colonies_Clean'], errors='coerce')

# Drop rows where either column couldn't be parsed
df = df.dropna(subset=['Loss', 'Colonies_Clean'])

print('Rows remaining after cleaning:', len(df))
print()
print(df[['State', 'Colonies_Clean', 'Loss']].head(10))

In [ ]:
# Plot the distribution of winter loss across all state-year observations
plt.figure(figsize=(10, 5))
sns.histplot(df['Loss'], bins=30, color='goldenrod', edgecolor='black')
plt.axvline(x=df['Loss'].mean(), color='red', linestyle='--', linewidth=2,
            label=f'Mean loss ({df["Loss"].mean():.1f}%)')
plt.xlabel('Winter Colony Loss (%)')
plt.ylabel('Number of Observations')
plt.title('Distribution of Winter Bee Colony Loss Across U.S. States')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Average winter loss: {df["Loss"].mean():.1f}%')
print(f'Median winter loss:  {df["Loss"].median():.1f}%')
print(f'Min:  {df["Loss"].min():.1f}%')
print(f'Max:  {df["Loss"].max():.1f}%')

## Step 2 — Choosing Our Target and Feature

To use Linear Regression we need:
- One **feature** (input) — the number we will use to make predictions
- One **target** (output) — the number we are trying to predict

We chose **number of colonies** as our feature and **winter loss %** as our target.

### Why winter loss %?

Honey bees are responsible for pollinating roughly one-third of the food we eat. Tracking the percentage of colonies lost each winter is the most widely used indicator of bee population health. A high loss percentage signals environmental stress, disease, or pesticide exposure — all serious concerns for food security.

### Why number of colonies as the predictor?

It's a natural question: does scale change outcomes? Do states with massive beekeeping operations manage winter loss better (more resources, professional management) or worse (disease spreads faster in dense populations)? Below we check whether a linear relationship between these two variables even exists before committing to this model.

In [ ]:
# Scatterplot — is there any visible linear trend?
plt.figure(figsize=(10, 6))
plt.scatter(df['Colonies_Clean'], df['Loss'],
            color='goldenrod', edgecolors='black', alpha=0.5)
plt.xlabel('Number of Colonies in State', fontsize=13)
plt.ylabel('Winter Colony Loss (%)', fontsize=13)
plt.title('Colonies vs. Winter Loss — Is There a Linear Trend?', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Show the range of colony counts across states
print('Colony count summary:')
print(f'  Min:    {df["Colonies_Clean"].min():,.0f}')
print(f'  Max:    {df["Colonies_Clean"].max():,.0f}')
print(f'  Mean:   {df["Colonies_Clean"].mean():,.0f}')
print(f'  Median: {df["Colonies_Clean"].median():,.0f}')
print()
print('Winter loss summary:')
print(f'  Min:  {df["Loss"].min():.1f}%')
print(f'  Max:  {df["Loss"].max():.1f}%')
print(f'  Mean: {df["Loss"].mean():.1f}%')

## Step 3 — Preprocessing

Before training we need to:
1. **Select our feature and target** — pull out the two columns we care about
2. **Reshape X to 2D** — our model expects a table, even with one column
3. **Split into train and test sets** — so we can evaluate fairly
4. **Normalize the features** — colony counts reach into the hundreds of thousands while loss % stays under 100; this scale difference confuses gradient descent

In [ ]:
# Step 1: Pull out feature (X) and target (y) as numpy arrays
X = df['Colonies_Clean'].values   # our input — number of colonies
y = df['Loss'].values             # our target — winter loss %

# Step 2: Reshape X from 1D (557,) to 2D (557, 1)
# Our LinearRegression class expects a 2D input — think of it as a table with one column
X = X.reshape(-1, 1)

print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
# Step 3: Split into train and test sets (80% train, 20% test)
# We shuffle first so the split is random, not ordered by state name
np.random.seed(42)  # set seed so results are the same every time you run this

indices = np.arange(len(X))
np.random.shuffle(indices)

split = int(0.8 * len(X))

X_train = X[indices[:split]]
X_test  = X[indices[split:]]
y_train = y[indices[:split]]
y_test  = y[indices[split:]]

print(f'Training set: {len(X_train)} observations')
print(f'Test set:     {len(X_test)} observations')

In [ ]:
# Step 4: Normalize the features using min-max scaling
#
# Formula: x_norm = (x - x_min) / (x_max - x_min)
# This squishes all values into the range [0, 1]
#
# IMPORTANT: We compute min and max from TRAINING data only.
# We then apply those same values to the test data.
# Never let test data influence preprocessing — that would be leaking
# information the model shouldn't have yet.

X_min = X_train.min()
X_max = X_train.max()

X_train_norm = (X_train - X_min) / (X_max - X_min)
X_test_norm  = (X_test  - X_min) / (X_max - X_min)

print('Before normalization — X_train min:', X_train.min(), '| max:', X_train.max())
print('After normalization  — X_train min:', round(X_train_norm.min(), 3), '| max:', round(X_train_norm.max(), 3))
print()
print('Features have been normalized to [0, 1]')

## Step 4 — Training the Model

Now the fun part! We import our custom `LinearRegression` class and train it on the bee colony data.

Remember the update rules from class — at each step, gradient descent nudges the weight and bias in the direction that reduces error:

$$w \leftarrow w - \alpha \cdot (\hat{y}^{(i)} - y^{(i)}) \cdot x^{(i)}$$
$$b \leftarrow b - \alpha \cdot (\hat{y}^{(i)} - y^{(i)})$$

The model will loop through all training examples `epochs` times, updating its weight and bias after every single example. This is called **Stochastic Gradient Descent (SGD)**.

In [ ]:
# Initialize our custom LinearRegression model
model = LinearRegression(learning_rate=0.01, epochs=100)

# Train it on the normalized training data
model.fit(X_train_norm, y_train)

print('Training complete!')
print(f'Learned weight (w): {model.weights[0]:.4f}')
print(f'Learned bias   (b): {model.bias:.4f}')
print()
print(f'Our model learned the line:  y_hat = {model.weights[0]:.2f} * x + {model.bias:.2f}')

In [ ]:
# Plot the learning curve — how does MSE change each epoch?
# If learning is working, this should go DOWN over time!
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(model.loss_history) + 1),
         model.loss_history,
         color='steelblue', linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error (MSE)')
plt.title('Linear Regression Learning Curve — MSE Per Epoch')
plt.tight_layout()
plt.show()

print('Each epoch is one full pass through all training examples.')
print('As the model learns, the error gets smaller and smaller!')

## Step 5 — Evaluation

Now we evaluate how well our model learned to predict winter colony loss.

We use three approaches:
- **MSE and R²** — numerical scores on train and test sets
- **Regression line plot** — visualize how well the line fits the data
- **Learning rate experiments** — see how our hyperparameter choice affects training

In [ ]:
# Evaluate on training data
train_mse = model.mean_squared_error(X_train_norm, y_train)
train_r2  = model.r_squared(X_train_norm, y_train)

# Evaluate on test data (data the model has never seen)
test_mse = model.mean_squared_error(X_test_norm, y_test)
test_r2  = model.r_squared(X_test_norm, y_test)

print('=== Training Set ===')
print(f'  MSE: {train_mse:.4f}  (lower is better)')
print(f'  R²:  {train_r2:.4f}  (closer to 1.0 is better)')
print()
print('=== Test Set ===')
print(f'  MSE: {test_mse:.4f}')
print(f'  R²:  {test_r2:.4f}')
print()
print('R² = 1.0 → perfect predictions')
print('R² = 0.0 → no better than always predicting the mean')
print('R² < 0   → worse than predicting the mean (something went wrong)')

In [ ]:
# Plot the regression line on top of the test data

# Create a smooth range of normalized x values for drawing the line
x_line_norm = np.linspace(0, 1, 200).reshape(-1, 1)
y_line = model.predict(x_line_norm)

# Convert normalized x back to original colony counts for axis labels
x_line_original = x_line_norm * (X_max - X_min) + X_min

plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test,
            color='goldenrod', edgecolors='black', alpha=0.6, label='Test data points')
plt.plot(x_line_original, y_line,
         color='firebrick', linewidth=2.5, label='Regression line')
plt.xlabel('Number of Colonies in State', fontsize=13)
plt.ylabel('Winter Colony Loss (%)', fontsize=13)
plt.title('Linear Regression — Colonies vs. Winter Loss', fontsize=15)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# How does the choice of learning rate affect training?
learning_rates = [0.001, 0.01, 0.1, 0.5]

fig, axs = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Effect of Learning Rate on Training Loss', fontsize=16)

for ax, lr in zip(axs.flat, learning_rates):
    m = LinearRegression(learning_rate=lr, epochs=100)
    m.fit(X_train_norm, y_train)
    ax.plot(range(1, len(m.loss_history) + 1), m.loss_history,
            color='steelblue', linewidth=2)
    ax.set_title(f'Learning Rate = {lr}', fontsize=13)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('MSE', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Make a prediction on a brand new, unseen colony count
# Remember: normalize using the SAME min/max as training!

new_colony_count = 50000
new_normalized = (new_colony_count - X_min) / (X_max - X_min)
new_normalized = np.array([[new_normalized]])

predicted_loss = model.predict(new_normalized)

print(f'A state with {new_colony_count:,} colonies is predicted to have')
print(f'a winter colony loss of: {predicted_loss[0]:.2f}%')

## Step 6 — Conclusion

### What did we find?

Our Linear Regression model was trained to predict the **winter colony loss percentage** of a U.S. state based on its **colony count**, using 557 observations from the BIP/USDA dataset.

### What do the results tell us?

The learned weight $w$ tells us the direction of the relationship — a positive weight means states with more colonies tend to have slightly higher loss rates, while a negative weight would suggest the opposite. The regression line and R² score tell us how well colony count alone explains winter loss.

The learning curve shows our model steadily reducing its MSE across epochs — each pass through the data nudges the line closer to the true trend in the data.

### Why colony count as the predictor?

Colony count is a clean, consistently recorded variable across all states and years. It's also a meaningful one — the scale of beekeeping operations could plausibly affect winter survival through disease pressure, management quality, or resource allocation. Even if the effect is small, understanding it matters for bee conservation policy.

### Limitations of Linear Regression

1. **One feature only** — we're only using colony count. Winter loss is almost certainly affected by many other variables: temperature, pesticide use, disease rates, beekeeper experience. A more complete model would include all of these.
2. **Assumes linearity** — if the true relationship between colonies and loss is curved or complex, a straight line will miss it.
3. **Sensitive to outliers** — very large states (like California with 490,000+ colonies) can pull the regression line significantly. MSE squares the error, making large mistakes extra costly.

### What's next?

In the following notebooks we will explore more powerful algorithms — starting with Logistic Regression, which brings probability into the picture for classification problems!